# Step 3 — One complete ResNet34-UNet training step

This notebook uses synthetic coastlines, so it downloads no satellite data. Its purpose is to make the complete model-training mechanism visible before we scale up.

## 1. Install the cloned repository
Run this notebook after cloning the GitHub repository in Colab. The `ml` extra installs PyTorch, the UNet implementation, and later model dependencies.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/hriship618/coastline-image-segmentation.git"
repository = Path("/content/coastline-image-segmentation")
if not repository.exists():
    subprocess.run(["git", "clone", REPO_URL, str(repository)], check=True)
os.chdir(repository)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[ml]"], check=True)
source_directory = str(repository / "src")
if source_directory not in sys.path:
    sys.path.insert(0, source_directory)
print(f"Working from: {repository}")

## 2. Create a two-image batch
A model receives `[batch, channels, height, width]`. The target omits a channel dimension because each pixel stores one integer class ID.

In [ ]:
import torch
from coastlearn.synthetic import make_synthetic_coast

examples = [make_synthetic_coast(height=128, width=128, seed=seed) for seed in (7, 11)]
images = torch.stack([torch.from_numpy(image) for image, _ in examples])
masks = torch.stack([torch.from_numpy(mask) for _, mask in examples])
print("images:", images.shape, images.dtype)
print("masks: ", masks.shape, masks.dtype)

## 3. Build the baseline
The ResNet34 encoder compresses the image into semantic feature maps. The UNet decoder upsamples them and uses encoder skip connections to recover fine shoreline detail. `pretrained=False` keeps this smoke test independent of a weight download; real training will use ImageNet weights.

In [ ]:
from coastlearn.models import build_resnet34_unet

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = build_resnet34_unet(in_channels=5, num_classes=2, pretrained=False).to(device)
images = images.to(device)
masks = masks.to(device)
print("device:", device)
print("parameters:", f"{sum(p.numel() for p in model.parameters()):,}")

## 4. Inspect the forward pass
The output has two scores per pixel: one for land and one for water. These raw scores are called logits.

In [ ]:
with torch.no_grad():
    logits = model(images)
print("input: ", images.shape)
print("logits:", logits.shape, "# [batch, land/water, height, width]")

## 5. Perform one optimizer update
Cross-entropy penalizes the model when the correct class does not receive the largest score. Backpropagation computes gradients, and AdamW uses those gradients to update the weights.

In [ ]:
from coastlearn.training import build_cross_entropy_loss, train_one_batch

loss_function = build_cross_entropy_loss(ignore_index=255).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
result = train_one_batch(model, images, masks, optimizer, loss_function)
print(result)

## What just happened

1. Two five-band images entered the network.
2. ResNet34 converted pixels into hierarchical features.
3. UNet restored those features to pixel resolution.
4. The final layer produced land and water logits for each pixel.
5. Cross-entropy compared the logits with the masks.
6. Backpropagation calculated a gradient for every trainable parameter.
7. AdamW changed the parameters once.

Real training repeats this process over many batches and validates after each complete pass through the training set.